# Van Son - Xay dung Neural Network (MLP)

Setup: dung lai data tu phan cua ban (train_loader, test_loader, class_names).


In [6]:
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

current_dir = Path.cwd()
REPO_ROOT = current_dir if (current_dir/"Practice_1").exists() else current_dir.parents[1] if current_dir.name=="notebooks" else current_dir
DATA_DIR = REPO_ROOT / "data"

transform = transforms.Compose([transforms.ToTensor()])
datasets.FashionMNIST.mirrors = ["https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/"]

train_dataset = datasets.FashionMNIST(root=str(DATA_DIR), train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root=str(DATA_DIR), train=False, download=True, transform=transform)

BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
class_names = ["T-shirt/top","Trouser","Pullover","Dress","Coat","Sandal","Shirt","Sneaker","Bag","Ankle boot"]
print("Data ready:", len(train_dataset), "train /", len(test_dataset), "test")


Data ready: 60000 train / 10000 test


## Mo ta MLP - Kien truc `784 -> 128 -> 64 -> 10`

`28x28 -> Flatten -> 784 -> Linear -> 128 -> ReLU -> 64 -> ReLU -> 10 outputs`

- **Input Layer**: anh `28x28x1`, qua `Flatten` thanh vector 784.
- **Hidden Layer**: 2 lop an `Linear(784,128)` va `Linear(128,64)`.
- **Activation Function**: `ReLU` sau moi lop an, giup mang hoc duoc quan he phi tuyen.
- **Output Layer**: `Linear(64,10)` tra ve 10 logits, khong Softmax.


## Cell 11 - Define Neural Network

In [7]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.network = nn.Sequential(
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.network(x)


## Cell 12 - Create Model

In [8]:
model = NeuralNetwork().to(device)


## Cell 13 - Display Model

In [9]:
print(model)


NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (network): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)


## Kiem tra input/output shape & dem trainable parameters

In [10]:
images, labels = next(iter(train_loader))
images = images.to(device)

with torch.no_grad():
    logits = model(images)

print("Input shape :", images.shape)   # [batch, 1, 28, 28]
print("Output shape:", logits.shape)   # [batch, 10]

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")


Input shape : torch.Size([64, 1, 28, 28])
Output shape: torch.Size([64, 10])
Trainable parameters: 109,386


## Vi sao khong dung Softmax

Output la logits tho vi `nn.CrossEntropyLoss` da gop san `LogSoftmax + NLLLoss` khi train;
luc du doan chi can `argmax(logits)` la ra lop du doan (Softmax khong doi thu tu logit lon nhat).


## Ham save / load model

In [11]:
def save_model(model, path="fashion_mnist.pth"):
    torch.save(model.state_dict(), path)
    print("Saved to", path)

def load_model(path="fashion_mnist.pth", device=device):
    m = NeuralNetwork().to(device)
    m.load_state_dict(torch.load(path, map_location=device))
    m.eval()
    return m

save_model(model)
loaded_model = load_model()

with torch.no_grad():
    same = torch.allclose(model(images), loaded_model(images))
print("Output khop sau khi load lai:", same)


Saved to fashion_mnist.pth
Output khop sau khi load lai: True


## Summary

MLP `784->128->64->10` (Flatten, Linear+ReLU x2, Linear output). Input `[N,1,28,28]` -> Output `[N,10]`.
Khong dung Softmax vi CrossEntropyLoss da xu ly. Da co ham `save_model`/`load_model`, kiem tra load lai cho ket qua giong nhau.
